# AI Scheduler: Latency-Optimal Edge–Cloud Workload Scheduling

This notebook runs the AI Scheduler pipeline and reproduces the main results of the paper from the measured data.

| Module | What it does | Section |
|---|---|---|
| 1. Code Analysis | Reads a Python program's syntax tree **without running it** and extracts 13 features | 1 |
| 2. Network Check | Smoothed round-trip time (RTT) to the AWS EC2 cloud worker | 2 |
| 3. Prediction and Routing | Predicts run time on the edge and in the cloud; sends the program to whichever finishes first | 3–5 |
| 4. Execution and Report | Runs the program and reports the output, the node and the time taken | 7 |

**Data:** 633 Python programs (`test_scripts/`), each measured on a CPU-limited edge (Docker, half a CPU core) and on an
AWS EC2 c7i-flex.large in ap-south-1 over the real network (`data/ec2_c7i_flex/`). Models are evaluated with whole
program families held out of training.

**How to run:** *Runtime → Run all*. It takes about 2–3 minutes and needs no GPU.

## 0. Setup
Gets the project code (from GitHub, or from the submitted zip if it is uploaded) and installs the library versions used in the paper.

In [ ]:
import os, subprocess, sys, glob, zipfile

REPO = "https://github.com/Adithya-109/ai-scheduler-aws.git"
COMMIT = "e33ab5b25e84c9139af20c89a47cbb67e312271a"   # version used for the paper

if not os.path.exists("pipeline/feature_extractor.py"):
    zips = [z for z in glob.glob("/content/*.zip") + glob.glob("*.zip") if "A2" in z or "Scheduler" in z]
    if zips:                                   # the submitted zip was uploaded to Colab
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall("/content/project")
        root = next(os.path.dirname(p) for p in glob.glob("/content/project/**/pipeline/feature_extractor.py", recursive=True))
        os.chdir(os.path.dirname(root))
    else:                                      # otherwise clone the public repository
        if not os.path.exists("ai-scheduler-aws"):
            subprocess.run(["git", "clone", "-q", REPO], check=True)
            subprocess.run(["git", "-C", "ai-scheduler-aws", "checkout", "-q", COMMIT], check=True)
        os.chdir("ai-scheduler-aws")
sys.path.insert(0, os.getcwd())
print("Project folder:", os.getcwd())

# scikit-learn 1.9.1 is the version the paper's results were produced with
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.9.1"], capture_output=True, text=True)
import sklearn
print("scikit-learn", sklearn.__version__, "" if sklearn.__version__ == "1.9.1" else
      "(1.9.1 not available here: numbers may differ slightly from the paper)")

In [ ]:
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (confusion_matrix, matthews_corrcoef, f1_score, precision_score,
                             recall_score, balanced_accuracy_score, r2_score)

from pipeline.feature_extractor import CodeFeatureExtractor, FEATURE_COLUMNS
from pipeline.dataset import load_benchmark
from pipeline.regressor import LinearPlusTreesRegressor
from pipeline.rtt import RttTracker

show = globals().get("display", print)
pd.set_option("display.width", 200, "display.max_columns", 20)
EPS = 1e-6
CSV, META = "data/ec2_c7i_flex/asym_benchmark.csv", "data/ec2_c7i_flex/asym_benchmark.meta.json"
meta = json.load(open(META))
RTT = float(meta["rtt_seconds"])
print(f"{len(FEATURE_COLUMNS)} features:", ", ".join(FEATURE_COLUMNS))

## 1. Code Analysis (zero-execution feature extraction)
The extractor walks the abstract syntax tree and estimates the program's work in log space: interpreted Python
operations, NumPy element operations and BLAS floating-point operations (Algorithm 1). Nothing is executed.

In [ ]:
ex = CodeFeatureExtractor()
demo = ["numpy_matmul_2000.py", "blassvd_1000.py", "sort_1000000.py", "fib_35.py", "string_50000.py", "script_001.py"]
cols = ["num_lines", "num_loops", "max_loop_depth", "has_heavy_lib", "has_recursion",
        "log_python_ops", "log_numpy_elems", "log_blas_flops", "estimated_complexity"]
rows = []
for f in demo:
    src = open(f"test_scripts/{f}", encoding="utf-8").read()
    t0 = time.perf_counter(); feats = ex.analyze(src); ms = (time.perf_counter() - t0) * 1000
    rows.append({"program": f, **{c: round(feats[c], 2) for c in cols}, "analysis time (ms)": round(ms, 2)})
show(pd.DataFrame(rows))
print("\nExample program (numpy_matmul_2000.py):\n" + open("test_scripts/numpy_matmul_2000.py").read())

## 2. Network Check
Round-trip time from the edge to the EC2 worker in ap-south-1, measured during data collection (21 probes).
During operation every `/health` probe updates an exponentially weighted moving average with α = 0.3 (Algorithm 2).

In [ ]:
samples = np.array(meta["rtt_samples_seconds"]) * 1000
print(f"Median RTT {np.median(samples):.1f} ms, interquartile range {np.percentile(samples, 25):.0f}-"
      f"{np.percentile(samples, 75):.0f} ms, 90th percentile {np.percentile(samples, 90):.0f} ms")
tracker = RttTracker(alpha=0.3)
ewma = [tracker.update(s) for s in samples]
plt.figure(figsize=(7, 2.8))
plt.plot(samples, "o-", label="probe"); plt.plot(ewma, "-", lw=2, label="smoothed (EWMA, α = 0.3)")
plt.xlabel("probe"); plt.ylabel("RTT (ms)"); plt.title("Edge → EC2 round-trip time"); plt.legend(); plt.tight_layout(); plt.show()

## 3. Dataset
633 programs; each was run three times on each node and the median kept. Cloud results are scored on the measured end-to-end time (compute + network).

In [ ]:
df = load_benchmark(CSV)
e = df.actual_edge_time.to_numpy()          # edge run time (sec)
c = df.actual_cloud_time.to_numpy()         # cloud compute time (sec)
cw = df.cloud_wall_time.to_numpy()          # cloud end-to-end time, measured from the edge (sec)
truth = cw < e                              # True where the cloud really was faster
print(f"{len(df)} programs in {df.family.nunique()} families; {(~truth).sum()} faster at the edge, {truth.sum()} faster in the cloud")
fam = (df.assign(cloud_faster=truth).groupby("family")
         .agg(scripts=("file", "size"), cloud_faster=("cloud_faster", "sum"), median_edge_sec=("actual_edge_time", "median"))
         .sort_values("scripts", ascending=False))
show(fam.head(15))

## 4. Prediction and Routing
Two regressors (edge and cloud) are trained on log10 run time. Each one is a robust (Huber) linear model on the work
estimates plus gradient-boosted trees on its residual, so predictions extrapolate to heavier programs (Eq. 4).
Routing rule (Eq. 2): **CLOUD if predicted cloud time + RTT < predicted edge time, else EDGE.**

Grouped 5-fold cross-validation: every prediction is for a program family the model has **not** seen.

In [ ]:
X = df[FEATURE_COLUMNS].to_numpy(float)
pe, pc = np.zeros(len(df)), np.zeros(len(df))
t0 = time.perf_counter()
for tr, te in GroupKFold(5).split(X, groups=df.family):
    pe[te] = 10 ** LinearPlusTreesRegressor().fit(X[tr], np.log10(e[tr] + EPS)).predict(X[te]) - EPS
    pc[te] = 10 ** LinearPlusTreesRegressor().fit(X[tr], np.log10(c[tr] + EPS)).predict(X[te]) - EPS
print(f"trained 10 models in {time.perf_counter() - t0:.1f} s")
print(f"R² (log space, families held out): edge {r2_score(np.log10(e + EPS), np.log10(pe + EPS)):.3f}, "
      f"cloud {r2_score(np.log10(c + EPS), np.log10(pc + EPS)):.3f}")
route_cloud = pc + RTT < pe                 # the routing decision for every program

## 5. Routing accuracy and turnaround: existing versus proposed (Tables IV and VII)

In [ ]:
def static_threshold(df, rtt):
    """Existing method: offload when the static cost estimate exceeds a threshold tuned on the training folds."""
    cx = df.estimated_complexity.to_numpy(float); cost_c = c + rtt
    out = np.zeros(len(df), bool)
    for tr, te in GroupKFold(5).split(cx, groups=df.family):
        cands = np.concatenate([[np.inf], np.unique(cx[tr])])
        tau = cands[int(np.argmin([np.where(cx[tr] >= t, cost_c[tr], e[tr]).sum() for t in cands]))]
        out[te] = cx[te] >= tau
    return out

policies = {"Edge-only": np.zeros(len(df), bool), "Cloud-only": np.ones(len(df), bool),
            "Static threshold": static_threshold(df, RTT), "AI Scheduler (Proposed)": route_cloud}
rows = {}
for name, ch in policies.items():
    lat = np.where(ch, cw, e)
    z = {"zero_division": 0}
    rows[name] = {"Routing accuracy (%)": 100 * (ch == truth).mean(),
                  "Precision, cloud (%)": 100 * precision_score(truth, ch, **z),
                  "Recall, cloud (%)": 100 * recall_score(truth, ch, **z),
                  "F1-score": f1_score(truth, ch, **z),
                  "MCC": matthews_corrcoef(truth, ch) if 0 < ch.sum() < len(ch) else 0.0,
                  "Misrouting rate (%)": 100 * (ch != truth).mean(),
                  "Mean turnaround per job (sec)": lat.mean(),
                  "Reduction vs edge-only (%)": 100 * (1 - lat.sum() / e.sum())}
table7 = pd.DataFrame(rows).round(3)
show(table7)

tn, fp, fn, tp = confusion_matrix(truth, route_cloud).ravel()
oracle = np.minimum(e, cw); sched = np.where(route_cloud, cw, e)
print(f"\nConfusion matrix (rows = really faster, columns = routed to): edge row [{tn} {fp}], cloud row [{fn} {tp}]")
print(f"Balanced accuracy {100 * balanced_accuracy_score(truth, route_cloud):.1f} %; "
      f"{100 * (sched.sum() / cw.sum() - 1):+.1f} % vs cloud-only; {100 * (sched.sum() / oracle.sum() - 1):.1f} % from the oracle")

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"width_ratios": [1, 2.2]})
M = np.array([[tn, fp], [fn, tp]])
a.imshow(M / M.sum(1, keepdims=True), cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        a.text(j, i, M[i, j], ha="center", va="center", fontsize=13, color="white" if M[i, j] > 300 or (i == 1 and j == 1) else "black")
a.set_xticks([0, 1], ["Edge", "Cloud"]); a.set_yticks([0, 1], ["Edge", "Cloud"])
a.set_xlabel("Routed to"); a.set_ylabel("Actually faster"); a.set_title("(a) Confusion matrix")
names = list(rows); x = np.arange(len(names)); w = 0.27
for k, m in enumerate(["Routing accuracy (%)", "Precision, cloud (%)", "Recall, cloud (%)"]):
    b.bar(x + (k - 1) * w, [rows[n][m] for n in names], w, label=m)
b.set_xticks(x, names); b.set_ylabel("%"); b.set_title("(b) Existing versus proposed"); b.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Single-file and multi-file submission (Tables V and VI)

In [ ]:
pick = ["numpy_matmul_2000.py", "blassvd_1000.py", "nested_2000x3000.py", "fib_35.py", "sort_1000000.py",
        "dict_1000000.py", "fib_33.py", "string_250000.py", "fib_24.py", "string_50000.py", "script_001.py"]
idx = {f: i for i, f in enumerate(df.file)}
t5 = pd.DataFrame([{"program": f, "predicted edge (sec)": pe[i], "predicted cloud + RTT (sec)": pc[i] + RTT,
                    "routed to": "Cloud" if route_cloud[i] else "Edge", "measured edge (sec)": e[i],
                    "measured cloud end-to-end (sec)": cw[i], "faster node": "Cloud" if truth[i] else "Edge",
                    "correct": "yes" if route_cloud[i] == truth[i] else "no"} for f in pick for i in [idx[f]]]).round(3)
show(t5)

# multi-file batches: random batches of 1-100 programs, files run one after another as in the dashboard
rng = np.random.default_rng(1)
lat = {"Edge-only": e, "Cloud-only": cw, "Static threshold": np.where(policies["Static threshold"], cw, e),
       "AI Scheduler (Proposed)": sched}
t6 = []
for k in (1, 5, 10, 25, 50, 100):
    idx_b = np.stack([rng.choice(len(e), k, replace=False) for _ in range(2000)])
    tot = {p: v[idx_b].sum(1) for p, v in lat.items()}
    best_fixed = np.minimum(tot["Edge-only"], tot["Cloud-only"])
    t6.append({"files per batch": k, **{f"{p} (sec)": tot[p].mean() for p in lat},
               "AI Scheduler vs edge-only (%)": 100 * (tot["AI Scheduler (Proposed)"].mean() / tot["Edge-only"].mean() - 1),
               "batches faster than edge-only (%)": 100 * (tot["AI Scheduler (Proposed)"] < tot["Edge-only"] - 1e-9).mean(),
               "batches faster than both fixed (%)": 100 * (tot["AI Scheduler (Proposed)"] < best_fixed - 1e-9).mean()})
show(pd.DataFrame(t6).round(2))

## 7. Try it: schedule your own program
The final models are trained on all 633 programs. Edit `my_code` and run the cell: the scheduler analyses the code
(without running it), predicts the time on each node, picks one, and then runs the program here to show the output.
(In the full system the program is sent to the chosen worker over REST; this notebook runs it locally.)

In [ ]:
edge_model = LinearPlusTreesRegressor().fit(X, np.log10(e + EPS))
cloud_model = LinearPlusTreesRegressor().fit(X, np.log10(c + EPS))

def schedule(code_str, rtt=RTT):
    feats = ex.analyze(code_str)
    if "error" in feats:
        return {"decision": "EDGE", "reason": feats["error"]}
    x = np.array([[feats[k] for k in FEATURE_COLUMNS]], float)
    t_edge = float(10 ** edge_model.predict(x)[0] - EPS)
    t_cloud = float(10 ** cloud_model.predict(x)[0] - EPS + rtt)
    return {"predicted edge (sec)": round(t_edge, 4), "predicted cloud + RTT (sec)": round(t_cloud, 4),
            "decision": "CLOUD" if t_cloud < t_edge else "EDGE"}

my_code = """
import numpy as np
a = np.random.rand(1500, 1500)
b = np.random.rand(1500, 1500)
print("checksum", float((a @ b).sum()))
"""
print(schedule(my_code))
t0 = time.perf_counter()
out = subprocess.run([sys.executable, "-c", my_code], capture_output=True, text=True, timeout=60)
print("output:", out.stdout.strip() or out.stderr.strip(), f"| ran here in {time.perf_counter() - t0:.2f} sec")
for f in ["string_50000.py", "sort_1000000.py", "fib_35.py"]:
    print(f, schedule(open(f"test_scripts/{f}").read()))

## Summary
* Code Analysis takes well under a millisecond per program and never runs the code.
* With program families held out, AI Scheduler (Proposed) sends about 95.6 % of programs to the faster node
  (MCC ≈ 0.81) against 85.3 % for the static-threshold method, and finishes the workload about 23 % faster than
  always running at the edge and within about 1.5 % of an oracle that knows every run time.
* The full system (FastAPI edge and cloud workers, Streamlit dashboard) is in the same repository: see `README.md`.